#Initialization

In [0]:
import uuid
from datetime import datetime

In [0]:
# dimension customer time (starts here...)

start_time = datetime.now()

In [0]:
%sql
USE CATALOG salesdatalakehouse;

## Read from Silver Schema, Prepare and Serve in Gold Schema

###Prepare dimension customers

In [0]:
display(
  spark.sql(
    '''
      CREATE OR REPLACE VIEW gold.dim_customers AS
      SELECT 
        ROW_NUMBER() OVER (ORDER BY ci.customer_id) AS customer_key,
        ci.customer_id,
        ci.customer_key AS customer_number,
        ci.first_name,
        ci.last_name,
        la.country,
        ci.marital_status,
        CASE WHEN ci.gender != 'n/a' THEN ci.gender ELSE COALESCE(ca.gender, 'n/a') END AS gender,
        ca.birth_date,
        ci.create_date
      FROM silver.crm_cust_info ci
      LEFT JOIN silver.erp_cust_az12 ca
      ON ci.customer_key = ca.customer_id
      LEFT JOIN silver.erp_loc_a101 la
      ON ci.customer_key = la.customer_id
    '''
  )
)

####dim_customers Logging

In [0]:

# dimension customers time (ends here...)

run_id = str(uuid.uuid4())
job_name = "sales_etl"
task_name = 'load_Gold'
schema_name = 'gold'
table_name = 'dim_customers'
row_inserted = spark.sql(''' SELECT COUNT(*) FROM salesdatalakehouse.gold.dim_customers''').collect()[0][0]
end_time = datetime.now()

# Logging into pipeline_runs table
spark.sql(f"""
  INSERT INTO salesdatalakehouse.audit.pipeline_runs
  VALUES (
    '{run_id}',
    '{job_name}',
    '{task_name}',
    '{schema_name}',
    '{table_name}',
    '{start_time}',
    '{end_time}',
    DATEDIFF(SECOND, '{start_time}', '{end_time}'),
    'SUCCESS',
    '{row_inserted}',
    0,
    0,
    NULL,
    CURRENT_TIMESTAMP()
  )
""")

print(f"✓ dimension customers info Logged with run {run_id}")

###Prepare dimension products

In [0]:
# dimension products time (starts here...)

start_time = datetime.now()

In [0]:
display(
    spark.sql(
      '''
        CREATE OR REPLACE VIEW gold.dim_products AS
        SELECT 
            ROW_NUMBER() OVER (ORDER BY pi.start_date, pi.product_key) AS product_key,
            pi.product_id,
            pi.product_key AS product_number,
            pi.product_name,
            pi.category_id,
            pc.category,
            pc.subcategory,
            pc.maintenance,
            pi.product_cost,
            pi.product_line,
            pi.start_date            
        FROM silver.crm_prd_info pi
        LEFT JOIN silver.erp_px_cat_g1v2 pc
        ON pi.category_id = pc.category_id
        WHERE pi.end_date IS NULL --Filter out Historical data
      '''
    )
)

####dim_products Logging

In [0]:

# dimension products time (ends here...)

run_id = str(uuid.uuid4())
job_name = "sales_etl"
task_name = 'load_Gold'
schema_name = 'gold'
table_name = 'dim_products'
row_inserted = spark.sql('''SELECT COUNT(*) FROM salesdatalakehouse.gold.dim_products''').collect()[0][0]
end_time = datetime.now()

# Logging into pipeline_runs table
spark.sql(f"""
  INSERT INTO salesdatalakehouse.audit.pipeline_runs
  VALUES (
    '{run_id}',
    '{job_name}',
    '{task_name}',
    '{schema_name}',
    '{table_name}',
    '{start_time}',
    '{end_time}',
    DATEDIFF(SECOND, '{start_time}', '{end_time}'),
    'SUCCESS',
    '{row_inserted}',
    0,
    0,
    NULL,
    CURRENT_TIMESTAMP()
  )
""")

print(f"✓ dimension products info Logged with run {run_id}")

###Prepare fact sales

In [0]:
# fact sales time (starts here...)

start_time = datetime.now()

In [0]:
display(
    spark.sql(
    '''
        CREATE OR REPLACE VIEW gold.fact_sales AS
        SELECT 
            sd.order_number,
            p.product_key,
            c.customer_key,
            sd.order_date,
            sd.ship_date,
            sd.due_date,
            sd.sales_amount,
            sd.quantity,
            sd.price
        FROM silver.crm_sales_details sd
        LEFT JOIN gold.dim_customers c
        ON sd.customer_id = c.customer_id
        LEFT JOIN gold.dim_products p
        ON sd.product_key = p.product_number
    '''
))

#### fact_sales Logging

In [0]:

# dimension products time (ends here...)

run_id = str(uuid.uuid4())
job_name = "sales_etl"
task_name = 'load_Gold'
schema_name = 'gold'
table_name = 'fact_sales'
row_inserted = spark.sql('''SELECT COUNT(*) FROM salesdatalakehouse.gold.fact_sales''').collect()[0][0]
end_time = datetime.now()

# Logging into pipeline_runs table
spark.sql(f"""
  INSERT INTO salesdatalakehouse.audit.pipeline_runs
  VALUES (
    '{run_id}',
    '{job_name}',
    '{task_name}',
    '{schema_name}',
    '{table_name}',
    '{start_time}',
    '{end_time}',
    DATEDIFF(SECOND, '{start_time}', '{end_time}'),
    'SUCCESS',
    '{row_inserted}',
    0,
    0,
    NULL,
    CURRENT_TIMESTAMP()
  )
""")

print(f"✓ dimension products info Logged with run {run_id}")